In [1]:
from pathlib import Path
import sys

# Locate the project folder from wherever the notebook is opened
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "model_setup.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_preparation import load_training_bundle
from src.model_setup import make_model_pipeline, shared_cv

X_train, y_train, metadata = load_training_bundle(ROOT)
cv = shared_cv(metadata)

print("Training Records:", X_train.shape[0])
print("Number of Features:", X_train.shape[1])
print("CV Folds:", cv.get_n_splits())

print("\nTarget Distribution:")
print(y_train.value_counts())

Training Records: 202795
Number of Features: 21
CV Folds: 5

Target Distribution:
HeartDiseaseorAttack
0    183698
1     19097
Name: count, dtype: int64


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV

# Create Logistic Regression model
lr_model = make_model_pipeline(
    LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    kind="linear"
)

# Define hyperparameter search space
lr_param_distributions = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "model__class_weight": [None, "balanced"],
    "model__solver": ["lbfgs", "liblinear"]
}

print("Logistic Regression Optimisation Model Created")
print("\nHyperparameter Search Space:")

for parameter, values in lr_param_distributions.items():
    print(f"{parameter}: {values}")

print("\nCross-Validation Folds:", cv.get_n_splits())
print("Primary Evaluation Metric: Average Precision")

Logistic Regression Optimisation Model Created

Hyperparameter Search Space:
model__C: [0.001, 0.01, 0.1, 1, 10, 100]
model__class_weight: [None, 'balanced']
model__solver: ['lbfgs', 'liblinear']

Cross-Validation Folds: 5
Primary Evaluation Metric: Average Precision


In [3]:
# Configure RandomizedSearchCV
lr_search = RandomizedSearchCV(
    estimator=lr_model,
    param_distributions=lr_param_distributions,
    n_iter=12,
    scoring="average_precision",
    cv=cv,
    random_state=42,
    n_jobs=1,
    verbose=1,
    refit=True,
    return_train_score=True
)

# Run hyperparameter optimisation
lr_search.fit(X_train, y_train)

# Display best results
print("\nLogistic Regression Optimisation Completed")
print("\nBest Hyperparameters:")

for parameter, value in lr_search.best_params_.items():
    print(f"{parameter}: {value}")

print(f"\nBest Cross-Validation Average Precision: {lr_search.best_score_:.4f}")

Fitting 5 folds for each of 12 candidates, totalling 60 fits

Logistic Regression Optimisation Completed

Best Hyperparameters:
model__solver: lbfgs
model__class_weight: None
model__C: 0.1

Best Cross-Validation Average Precision: 0.3689


In [4]:
import pandas as pd
from sklearn.model_selection import cross_validate

# Get the best Logistic Regression model
lr_optimised_model = lr_search.best_estimator_

# Evaluate using the same 5-fold cross-validation
scoring = {
    "Accuracy": "accuracy",
    "Precision": "precision",
    "Recall": "recall",
    "F1": "f1",
    "ROC_AUC": "roc_auc",
    "Average_Precision": "average_precision"
}

lr_optimised_cv = cross_validate(
    estimator=lr_optimised_model,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
    return_train_score=True
)

# Summarise validation performance
lr_optimised_summary = pd.DataFrame({
    "Metric": list(scoring.keys()),
    "Mean": [
        lr_optimised_cv[f"test_{metric}"].mean()
        for metric in scoring
    ],
    "Std": [
        lr_optimised_cv[f"test_{metric}"].std()
        for metric in scoring
    ]
})

print("Optimised Logistic Regression - 5-Fold CV Results")
display(lr_optimised_summary.round(4))

Optimised Logistic Regression - 5-Fold CV Results


,Metric,Mean,Std
0,Accuracy,0.9074,0.0012
1,Precision,0.5375,0.0186
2,Recall,0.1222,0.0056
3,F1,0.1991,0.0086
4,ROC_AUC,0.8465,0.0039
5,Average_Precision,0.3689,0.0066


In [ ]:
import pandas as pd

# Load baseline results
baseline_path = (
    ROOT / "results" / "member2" /
    "logistic_regression_baseline.csv"
)
lr_baseline = pd.read_csv(baseline_path)

# Prepare baseline and optimised results
baseline = lr_baseline.set_index("Metric")
optimised = lr_optimised_summary.set_index("Metric")

# Compare mean and standard deviation
lr_comparison = pd.DataFrame({
    "Metric": optimised.index,
    "Baseline Mean": baseline["Mean Validation Score"].reindex(optimised.index).values,
    "Baseline Std": baseline["Standard Deviation"].reindex(optimised.index).values,
    "Optimised Mean": optimised["Mean"].values,
    "Optimised Std": optimised["Std"].values
})

# Calculate performance difference
lr_comparison["Mean Change"] = (
    lr_comparison["Optimised Mean"] - lr_comparison["Baseline Mean"]
)

print("Logistic Regression - Baseline vs Optimised")
display(lr_comparison.round(4))

Logistic Regression - Baseline vs Optimised


,Metric,Baseline Mean,Baseline Std,Optimised Mean,Optimised Std,Mean Change
0,Accuracy,0.9074,0.0012,0.9074,0.0012,0.0000
1,Precision,0.5360,0.0192,0.5375,0.0186,0.0015
2,Recall,0.1236,0.0050,0.1222,0.0056,-0.0014
3,F1,0.2009,0.0078,0.1991,0.0086,-0.0018
4,ROC_AUC,0.8465,0.0039,0.8465,0.0039,-0.0000
5,Average_Precision,0.3687,0.0065,0.3689,0.0066,0.0002


In [6]:
# Compare optimised training and validation performance
lr_optimised_overfitting = pd.DataFrame({
    "Metric": ["Accuracy", "F1", "ROC_AUC", "Average_Precision"],
    "Training Score": [
        lr_optimised_cv[f"train_{metric}"].mean()
        for metric in ["Accuracy", "F1", "ROC_AUC", "Average_Precision"]
    ],
    "Validation Score": [
        lr_optimised_cv[f"test_{metric}"].mean()
        for metric in ["Accuracy", "F1", "ROC_AUC", "Average_Precision"]
    ]
})

# Calculate the performance gap
lr_optimised_overfitting["Gap"] = (
    lr_optimised_overfitting["Training Score"]
    - lr_optimised_overfitting["Validation Score"]
)

print("Optimised Logistic Regression - Training vs Validation")
display(lr_optimised_overfitting.round(4))

Optimised Logistic Regression - Training vs Validation


,Metric,Training Score,Validation Score,Gap
0,Accuracy,0.9075,0.9074,0.0001
1,F1,0.2002,0.1991,0.0011
2,ROC_AUC,0.8470,0.8465,0.0005
3,Average_Precision,0.3694,0.3689,0.0005


In [7]:
import pandas as pd

# Use the existing results folder
results_dir = ROOT / "results" / "member2"
results_dir.mkdir(parents=True, exist_ok=True)

# Save optimised evaluation results
lr_optimised_summary.to_csv(
    results_dir / "logistic_regression_optimised.csv",
    index=False
)

# Save baseline vs optimised comparison
lr_comparison.to_csv(
    results_dir / "logistic_regression_optimisation_comparison.csv",
    index=False
)

# Save training vs validation analysis
lr_optimised_overfitting.to_csv(
    results_dir / "logistic_regression_optimised_overfitting.csv",
    index=False
)

# Save best hyperparameters
lr_best_params = pd.DataFrame(
    list(lr_search.best_params_.items()),
    columns=["Hyperparameter", "Best Value"]
)

lr_best_params.to_csv(
    results_dir / "logistic_regression_best_parameters.csv",
    index=False
)

print("All Logistic Regression Optimisation Results Saved Successfully")

All Logistic Regression Optimisation Results Saved Successfully
